In [24]:
# importing the packages
import numpy as np
import sys
from pyomo.environ import *

In [25]:
solvername = "glpk"

solverpath_folder = "C:\\glpk-4.65\\w64"  # does not need to be directly on c drive

solverpath_exe = "C:\\glpk-4.65\\w64\\glpsol"  # does not need to be directly on c drive

In [26]:
def read_data(index=0):
    # reading the cost_matrix and the pair_matrix
    cost_matrix = np.loadtxt("../data/cost_matrix/cost.txt").reshape(-1, 1)
    pair_matrix = np.genfromtxt(
        "../data/pairings/pair_array.txt", delimiter=",", dtype="int"
    )
    if index != 0:
        cost_matrix = cost_matrix[index]
        pair_matrix = pair_matrix[index]
    # determining the number of flights and tasks
    num_pairs = pair_matrix.shape[0]
    num_flights = pair_matrix.shape[1]
    print(num_pairs, num_flights)

    return cost_matrix, pair_matrix, num_pairs, num_flights


cost_matrix, pair_matrix, num_pairs, num_flights = read_data()

261693 160


In [28]:
# Initializing the Pyomo model
model = ConcreteModel()

# creating the binary allocation variable
model.x = Var(range(num_pairs), within=Binary)

# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
model.constraints = ConstraintList()
for i in range(num_flights):
    model.constraints.add(
        expr=sum(model.x[j] * pair_matrix[j, i] for j in range(num_pairs)) >= 1
    )

# declaring the objective function
model.obj = Objective(
    expr=sum(cost_matrix[j] * model.x[j] for j in range(num_pairs)), sense=minimize
)

model.dual = Suffix(direction=Suffix.IMPORT)

# Create a solver
solver = SolverFactory(solvername, executable=solverpath_exe)

# Solve the model
results = solver.solve(model)

# Check the solver status and termination condition
if (
    results.solver.status == SolverStatus.ok
    and results.solver.termination_condition == TerminationCondition.optimal
):
    print("The Solution is OPTIMAL")
elif results.solver.termination_condition == TerminationCondition.feasible:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

# Access the optimal variable values
x_values = [model.x[j].value for j in range(num_pairs)]
index = [j for j in range(num_pairs) if x_values[j] != 0]
print(value(model.obj))
# Access the dual values (shadow prices)
# dual_values = [model.dual[model.constraints[i]] for i in model.constraints]

The Solution is OPTIMAL
1009.0


In [29]:
index

[461,
 1458,
 3174,
 4322,
 5251,
 7533,
 7687,
 8035,
 8100,
 9424,
 10620,
 11042,
 12658,
 13157,
 15375,
 16034,
 16215,
 16723,
 17184,
 17610,
 18724,
 18808,
 18815,
 19536,
 19593,
 19595,
 19945,
 20016,
 20224,
 20356,
 41085,
 49001,
 78079,
 90206,
 108566,
 138550,
 141111,
 141272,
 142781,
 142890,
 195882,
 257800]

In [ ]:
# The Solution is OPTIMAL
# [0, 9, 19, 48, 49, 54, 57, 60, 61, 63, 64, 66, 75, 95, 103, 112, 121, 126, 137, 144, 148, 171, 192, 198, 206, 215, 219, 251, 258, 267, 277, 281, 285, 287, 290, 299, 310, 317, 334, 340, 344, 353, 368, 374, 380, 386, 388, 394, 409, 420, 436, 452, 456, 462, 468, 469, 481, 485, 488, 495, 506, 535, 536, 551, 563, 570, 574, 577, 591, 597, 599, 605, 614, 619, 624, 629, 633, 640, 647, 657, 659, 664, 670, 674, 694]
# 270.0